In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial.distance import pdist, squareform
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

df = pd.read_csv("cleveland.csv")

df["age_s"] = (df["age"] - df["age"].mean()) / df["age"].std()
# sex
df["trestbps_s"] = (df["trestbps"] - df["trestbps"].mean()) / df["trestbps"].std()
df["chol_s"] = (df["chol"] - df["chol"].mean()) / df["chol"].std()
df["fbs"] = df["fbs"].astype(bool)
# restecg
df["thalach_s"] = (df["thalach"] - df["thalach"].mean()) / df["thalach"].std()
df["exang"] = df["exang"].astype(bool)
df["oldpeak_s"] = (df["oldpeak"] - df["oldpeak"].mean()) / df["oldpeak"].std()
# slope
df["ca"] = df["ca"].replace("?", pd.NA).astype(np.float64)
df["ca_s"] = (df["ca"] - df["ca"].mean()) / df["ca"].std()
# thal
df["num"] = df["num"] != 0

df

In [ ]:
df_filtered = df[["age_s", "trestbps_s", "chol_s", "thalach_s", "oldpeak_s", "ca_s", "num"]].copy()
df_filtered.dropna(inplace=True)

X = df_filtered.drop(columns=["num"])
y = df_filtered["num"].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pairwise_distances = pdist(X.values, metric='euclidean')
dist_matrix = squareform(pairwise_distances)
np.fill_diagonal(dist_matrix, np.inf)
nearest_neighbors = np.argsort(dist_matrix, axis=1)

k=5
knn_lookup = nearest_neighbors[:, :k]
knn_labeled = y[knn_lookup]

In [ ]:
from sklearn.base import BaseEstimator, ClassifierMixin
from scipy.spatial.distance import cdist
import numpy as np

class KnnClassifier(ClassifierMixin, BaseEstimator):
    def __init__(self, k=5):
        self.k = k

    def fit(self, X, y):
        self.X = np.asarray(X)
        self.y = np.asarray(y)
        self.classes_ = np.unique(self.y)
        return self

    def predict(self, X):
        dist_matrix = cdist(np.asarray(X), self.X)
        nearest_neighbors = np.argsort(dist_matrix, axis=1)
        knn_lookup = nearest_neighbors[:, :self.k]
        knn_labeled = self.y[knn_lookup]
        return np.mean(knn_labeled, axis=1) >= 0.5

    def score(self, X, y, sample_weight=None):
        correct = self.predict(X) == np.asarray(y)
        return np.average(correct)

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    "k": range(1, 21)
}

grid_search = GridSearchCV(
    estimator=KnnClassifier(),
    param_grid=param_grid,
    cv=5,
    scoring="accuracy",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

results = pd.DataFrame(grid_search.cv_results_)

plt.plot(
    results["param_k"],
    results["mean_test_score"],
    marker="o"
)

plt.xlabel("Number of neighbors (k)")
plt.ylabel("Cross-validation accuracy")
plt.title("KNN Hyperparameter Tuning")
plt.show()